In [6]:
import requests
import urllib3
import pandas as pd
from bs4 import BeautifulSoup
import re
import time
from pathlib import Path

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [7]:
class RunesDBParser:
    """
    Парсер для базы данных рунических надписей runesdb.eu
    """

    def __init__(self):
        self.base_url = "https://www.runesdb.eu"
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
        })
        self.images_dir = Path("runic_images")
        self.images_dir.mkdir(exist_ok=True)

    def get_findlist(self, country=None, findplace_starts_with=None, max_results=100):
        """
        Получить список находок через веб-интерфейс

        Параметры:
        - country: код страны (DE, SE, NO и т.д.)
        - findplace_starts_with: начало названия места находки
        - max_results: максимальное количество результатов
        """

        # Пробуем разные варианты URL
        urls_to_try = [
            f"{self.base_url}/en/find-list",
            f"{self.base_url}/findlist",
            f"{self.base_url}/en/findlist"
        ]

        for url in urls_to_try:
            try:
                print(f"Пробуем URL: {url}")
                response = self.session.get(url, verify=False, timeout=30)

                if response.status_code == 200:
                    print(f"✓ Успешно загружено: {url}")
                    soup = BeautifulSoup(response.text, 'html.parser')

                    # Ищем ссылки на отдельные находки
                    find_links = []
                    for link in soup.find_all('a', href=True):
                        href = link['href']
                        # Ищем ссылки вида /find/ID или /findlist/...
                        if '/find/' in href or re.match(r'.*/findlist/.*/f/\d+', href):
                            full_url = href if href.startswith('http') else f"{self.base_url}{href}"
                            find_links.append(full_url)

                    print(f"Найдено {len(find_links)} ссылок на находки")
                    return find_links[:max_results]

            except Exception as e:
                print(f"✗ Ошибка для {url}: {e}")
                continue

        print("⚠ Не удалось загрузить список находок через веб-интерфейс")
        return []

    def parse_find_page(self, url):
        """
        Парсинг страницы отдельной находки
        """
        try:
            response = self.session.get(url, verify=False, timeout=30)
            soup = BeautifulSoup(response.text, 'html.parser')

            data = {
                'url': url,
                'find_id': re.search(r'/f/(\d+)', url).group(1) if re.search(r'/f/(\d+)', url) else None,
                'images': [],
                'transliteration': None,
                'translation': None,
                'runerow': None,
                'findplace': None,
                'country': None,
                'object_class': None,
                'dating': None,
                'inscription': None
            }

            # Извлекаем изображения
            for img in soup.find_all('img'):
                img_src = img.get('src', '')
                if 'upload' in img_src or 'image' in img_src:
                    img_url = img_src if img_src.startswith('http') else f"{self.base_url}{img_src}"
                    data['images'].append(img_url)

            # Извлекаем текстовые данные
            text = soup.get_text()

            # Ищем транслитерацию
            translit_match = re.search(r'Transliteration[:\s]+([^\n]+)', text, re.I)
            if translit_match:
                data['transliteration'] = translit_match.group(1).strip()

            # Ищем перевод
            transl_match = re.search(r'Translation[:\s]+([^\n]+)', text, re.I)
            if transl_match:
                data['translation'] = transl_match.group(1).strip()

            # Ищем тип рунического ряда
            runerow_match = re.search(r'Runerow[:\s]+([^\n]+)', text, re.I)
            if runerow_match:
                data['runerow'] = runerow_match.group(1).strip()

            # Ищем место находки
            findplace_match = re.search(r'Find[\s-]?place[:\s]+([^\n]+)', text, re.I)
            if findplace_match:
                data['findplace'] = findplace_match.group(1).strip()

            # Ищем страну
            country_match = re.search(r'Country[:\s]+([^\n]+)', text, re.I)
            if country_match:
                data['country'] = country_match.group(1).strip()

            # Ищем класс объекта
            objclass_match = re.search(r'Object[\s-]?class[:\s]+([^\n]+)', text, re.I)
            if objclass_match:
                data['object_class'] = objclass_match.group(1).strip()

            # Ищем датировку
            dating_match = re.search(r'Dating[:\s]+([^\n]+)', text, re.I)
            if dating_match:
                data['dating'] = dating_match.group(1).strip()

            return data

        except Exception as e:
            print(f"Ошибка при парсинге {url}: {e}")
            return None

    def download_image(self, img_url, find_id, img_index):
        """
        Скачивание изображения
        """
        try:
            response = self.session.get(img_url, verify=False, timeout=30)
            if response.status_code == 200:
                ext = img_url.split('.')[-1].split('?')[0]
                if ext not in ['jpg', 'jpeg', 'png', 'gif', 'webp']:
                    ext = 'jpg'

                filename = self.images_dir / f"{find_id}_{img_index}.{ext}"
                with open(filename, 'wb') as f:
                    f.write(response.content)
                return str(filename)
        except Exception as e:
            print(f"Ошибка загрузки изображения {img_url}: {e}")
        return None

    def scrape_manual(self, start_id=1, end_id=100, delay=2):
        """
        Ручной перебор ID находок (если API не работает)

        Параметры:
        - start_id: начальный ID
        - end_id: конечный ID
        - delay: задержка между запросами (секунды)
        """
        results = []

        # Пробуем разные форматы URL
        url_patterns = [
            f"{self.base_url}/en/findlist/d/fa/q////6/f/{{id}}",
            f"{self.base_url}/findlist/d/fa/q////6/f/{{id}}",
            f"{self.base_url}/find/{{id}}"
        ]

        for find_id in range(start_id, end_id + 1):
            print(f"\n[{find_id}/{end_id}] Обработка находки ID: {find_id}")

            success = False
            for pattern in url_patterns:
                url = pattern.format(id=find_id)

                try:
                    response = self.session.get(url, verify=False, timeout=30)
                    if response.status_code == 200:
                        data = self.parse_find_page(url)
                        if data and any(data.values()):
                            # Скачиваем изображения
                            # local_images = []
                            # for idx, img_url in enumerate(data['images']):
                            #     local_path = self.download_image(img_url, find_id, idx)
                            #     if local_path:
                            #         local_images.append(local_path)

                            # data['local_images'] = local_images
                            results.append(data)
                            print(f"✓ Найдена информация")
                            success = True
                            break
                except:
                    continue

            if not success:
                print(f"✗ Не удалось получить данные")

            time.sleep(delay)

        return results

    def create_dataframe(self, results):
        """
        Создание DataFrame из результатов
        """
        if not results:
            print("Нет данных для создания DataFrame")
            return pd.DataFrame()

        # Преобразуем списки изображений в строки
        for r in results:
            if 'images' in r:
                r['images_urls'] = '; '.join(r['images'])
                del r['images']
            if 'local_images' in r:
                r['local_images_paths'] = '; '.join(r['local_images'])
                del r['local_images']

        df = pd.DataFrame(results)
        return df


In [10]:
parser = RunesDBParser()

result = parser.scrape_manual(start_id=10000, end_id=12000, delay=1)
print("\n\n=== РЕЗУЛЬТАТЫ ===")
#закончил на 1178


[10000/12000] Обработка находки ID: 10000
✗ Не удалось получить данные

[10001/12000] Обработка находки ID: 10001
✓ Найдена информация

[10002/12000] Обработка находки ID: 10002
✓ Найдена информация

[10003/12000] Обработка находки ID: 10003
✓ Найдена информация

[10004/12000] Обработка находки ID: 10004
✓ Найдена информация

[10005/12000] Обработка находки ID: 10005
✗ Не удалось получить данные

[10006/12000] Обработка находки ID: 10006
✗ Не удалось получить данные

[10007/12000] Обработка находки ID: 10007
✗ Не удалось получить данные

[10008/12000] Обработка находки ID: 10008
✗ Не удалось получить данные

[10009/12000] Обработка находки ID: 10009
✗ Не удалось получить данные

[10010/12000] Обработка находки ID: 10010
✗ Не удалось получить данные

[10011/12000] Обработка находки ID: 10011
✗ Не удалось получить данные

[10012/12000] Обработка находки ID: 10012
✗ Не удалось получить данные

[10013/12000] Обработка находки ID: 10013
✗ Не удалось получить данные

[10014/12000] Обработка

KeyboardInterrupt: 

In [11]:
df2 = parser.create_dataframe(result)
print(df2.head())
output_file = "runes_parsed_parallel_10k+.csv"
df2.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n✓ Данные сохранены в: {output_file}")
print(f"✓ Изображения сохранены в папке: {parser.images_dir}")

                             url find_id       transliteration  \
0  https://www.runesdb.eu/find/1    None              hariso |   
1  https://www.runesdb.eu/find/2    None    [0-?](w)iduhudaz |   
2  https://www.runesdb.eu/find/3    None        alugod(0-1Z) |   
3  https://www.runesdb.eu/find/4    None  ekunwod(1-2?)[0-?] |   
4  https://www.runesdb.eu/find/5    None   (w)ara(2-3?)s(1?) |   

       translation runerow findplace country object_class     dating  \
0          Harisō.    None      None    None         None  210 - 310   
1  ... Widuhundaz.    None      None    None         None  210 - 260   
2          Alugōd.    None      None    None         None  210 - 240   
3    ich, Unwōd...    None      None    None         None  210 - 240   
4                -    None      None    None         None  210 - 240   

  inscription                                        images_urls  
0        None  https://www.runesdb.eu/image-service/2758/full...  
1        None  https://www.runesdb.e